In [12]:
import pandas as pd
from pathlib import Path

In [13]:
pm1 = pd.read_csv("../data/raw/ratnapark/pm1.csv")
pm10 = pd.read_csv("../data/raw/ratnapark/pm10.csv")
pm25 = pd.read_csv("../data/raw/ratnapark/pm25.csv")

print("PM1:", pm1.shape)
print("PM10:", pm10.shape)
print("PM2.5:", pm25.shape)

PM1: (19613, 2)
PM10: (19610, 2)
PM2.5: (19627, 2)


In [14]:
pm1["datetime"] = pd.to_datetime(pm1["datetime"], utc=True)
pm10["datetime"] = pd.to_datetime(pm10["datetime"], utc=True)
pm25["datetime"] = pd.to_datetime(pm25["datetime"], utc=True)

In [15]:
test_raw = (
    pm1
    .merge(
        pm10,
        on="datetime",
        how="outer"
    )
    .merge(
        pm25,
        on="datetime",
        how="outer"
    )
    .sort_values("datetime")
    .reset_index(drop=True)
)

display(test_raw.head())
display(test_raw.tail())

print("Shape:", test_raw.shape)

,datetime,pm1,pm10,pm25
0,2026-08-02 00:00:00+00:00,69.500000,84.500000,81.099998
1,2026-08-02 00:01:00+00:00,120.599998,134.000000,127.500000
2,2026-08-02 00:02:00+00:00,181.300003,195.899994,190.699997
3,2026-08-02 00:03:00+00:00,218.199997,231.500000,226.800003
4,2026-08-02 00:04:00+00:00,87.199997,100.800003,89.000000


,datetime,pm1,pm10,pm25
19637,2026-08-16 05:55:00+00:00,11.1,24.400000,14.9
19638,2026-08-16 05:56:00+00:00,8.8,27.900000,12.1
19639,2026-08-16 05:57:00+00:00,8.0,24.400000,10.2
19640,2026-08-16 05:58:00+00:00,7.3,16.799999,9.4
19641,2026-08-16 05:59:00+00:00,7.5,12.300000,8.4


Shape: (19642, 4)


In [16]:
print("Missing values:")
display(test_raw.isna().sum())

print("\nDate range:")
print(test_raw["datetime"].min())
print(test_raw["datetime"].max())

Missing values:


datetime     0
pm1         29
pm10        32
pm25        15
dtype: int64


Date range:
2026-08-02 00:00:00+00:00
2026-08-16 05:59:00+00:00


COnvert minute data to daily 

In [17]:
test_daily = (
    test_raw
    .set_index("datetime")
    .resample("D")
    .mean(numeric_only=True)
    .reset_index()
)

test_daily = test_daily.rename(
    columns={
        "datetime": "date",
        "pm25": "pm2_5"
    }
)

test_daily = (
    test_daily
    .sort_values("date")
    .reset_index(drop=True)
)

display(test_daily)

,date,pm1,pm10,pm2_5
0,2026-08-02 00:00:00+00:00,55.195013,64.022251,58.833538
1,2026-08-03 00:00:00+00:00,56.616693,75.393556,62.634817
2,2026-08-04 00:00:00+00:00,51.808746,69.281502,58.748684
3,2026-08-05 00:00:00+00:00,35.848899,49.294396,39.631938
4,2026-08-06 00:00:00+00:00,36.590483,49.773565,41.120997
5,2026-08-07 00:00:00+00:00,17.447983,31.665229,20.730946
6,2026-08-08 00:00:00+00:00,10.267731,20.642748,13.310965
7,2026-08-09 00:00:00+00:00,8.037196,17.865232,10.405691
8,2026-08-10 00:00:00+00:00,6.991672,16.479667,9.005760
9,2026-08-11 00:00:00+00:00,11.383264,26.751389,15.107986


In [18]:
print("Shape:", test_daily.shape)

print("\nDate range:")
print(test_daily["date"].min())
print(test_daily["date"].max())

print("\nMissing values:")
display(test_daily.isna().sum())

Shape: (15, 4)

Date range:
2026-08-02 00:00:00+00:00
2026-08-16 00:00:00+00:00

Missing values:


date     0
pm1      0
pm10     0
pm2_5    0
dtype: int64

In [19]:
for lag in range(1, 16):

    test_daily[f"pm2_5_lag_{lag}"] = (
        test_daily["pm2_5"].shift(lag)
    )

    test_daily[f"pm10_lag_{lag}"] = (
        test_daily["pm10"].shift(lag)
    )

In [20]:
for window in [3, 7, 15]:

    test_daily[f"pm2_5_rolling_mean_{window}"] = (
        test_daily["pm2_5"]
        .shift(1)
        .rolling(window)
        .mean()
    )

    test_daily[f"pm10_rolling_mean_{window}"] = (
        test_daily["pm10"]
        .shift(1)
        .rolling(window)
        .mean()
    )

In [21]:
test_features = test_daily[
    test_daily["date"].between(
        "2026-08-02",
        "2026-08-16"
    )
].copy()

test_features = (
    test_features
    .sort_values("date")
    .reset_index(drop=True)
)

print("Test rows:", len(test_features))

display(test_features)

Test rows: 15


,date,pm1,pm10,pm2_5,pm2_5_lag_1,pm10_lag_1,pm2_5_lag_2,pm10_lag_2,pm2_5_lag_3,pm10_lag_3,...,pm2_5_lag_14,pm10_lag_14,pm2_5_lag_15,pm10_lag_15,pm2_5_rolling_mean_3,pm10_rolling_mean_3,pm2_5_rolling_mean_7,pm10_rolling_mean_7,pm2_5_rolling_mean_15,pm10_rolling_mean_15
0,2026-08-02 00:00:00+00:00,55.195013,64.022251,58.833538,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-08-03 00:00:00+00:00,56.616693,75.393556,62.634817,58.833538,64.022251,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-08-04 00:00:00+00:00,51.808746,69.281502,58.748684,62.634817,75.393556,58.833538,64.022251,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-08-05 00:00:00+00:00,35.848899,49.294396,39.631938,58.748684,69.281502,62.634817,75.393556,58.833538,64.022251,...,NaN,NaN,NaN,NaN,60.072347,69.565770,NaN,NaN,NaN,NaN
4,2026-08-06 00:00:00+00:00,36.590483,49.773565,41.120997,39.631938,49.294396,58.748684,69.281502,62.634817,75.393556,...,NaN,NaN,NaN,NaN,53.671813,64.656485,NaN,NaN,NaN,NaN
5,2026-08-07 00:00:00+00:00,17.447983,31.665229,20.730946,41.120997,49.773565,39.631938,49.294396,58.748684,69.281502,...,NaN,NaN,NaN,NaN,46.500540,56.116488,NaN,NaN,NaN,NaN
6,2026-08-08 00:00:00+00:00,10.267731,20.642748,13.310965,20.730946,31.665229,41.120997,49.773565,39.631938,49.294396,...,NaN,NaN,NaN,NaN,33.827960,43.577730,NaN,NaN,NaN,NaN
7,2026-08-09 00:00:00+00:00,8.037196,17.865232,10.405691,13.310965,20.642748,20.730946,31.665229,41.120997,49.773565,...,NaN,NaN,NaN,NaN,25.054302,34.027181,42.144555,51.439035,NaN,NaN
8,2026-08-10 00:00:00+00:00,6.991672,16.479667,9.005760,10.405691,17.865232,13.310965,20.642748,20.730946,31.665229,...,NaN,NaN,NaN,NaN,14.815867,23.391070,35.226291,44.845176,NaN,NaN
9,2026-08-11 00:00:00+00:00,11.383264,26.751389,15.107986,9.005760,16.479667,10.405691,17.865232,13.310965,20.642748,...,NaN,NaN,NaN,NaN,10.907472,18.329216,27.564997,36.428906,NaN,NaN


In [22]:
print("Missing values in test features:")
display(test_features.isna().sum())

Missing values in test features:


date                      0
pm1                       0
pm10                      0
pm2_5                     0
pm2_5_lag_1               1
pm10_lag_1                1
pm2_5_lag_2               2
pm10_lag_2                2
pm2_5_lag_3               3
pm10_lag_3                3
pm2_5_lag_4               4
pm10_lag_4                4
pm2_5_lag_5               5
pm10_lag_5                5
pm2_5_lag_6               6
pm10_lag_6                6
pm2_5_lag_7               7
pm10_lag_7                7
pm2_5_lag_8               8
pm10_lag_8                8
pm2_5_lag_9               9
pm10_lag_9                9
pm2_5_lag_10             10
pm10_lag_10              10
pm2_5_lag_11             11
pm10_lag_11              11
pm2_5_lag_12             12
pm10_lag_12              12
pm2_5_lag_13             13
pm10_lag_13              13
pm2_5_lag_14             14
pm10_lag_14              14
pm2_5_lag_15             15
pm10_lag_15              15
pm2_5_rolling_mean_3      3
pm10_rolling_mean_3 

In [23]:
feature_columns = [
    col
    for col in test_features.columns
    if (
        col.startswith("pm2_5_lag_")
        or col.startswith("pm10_lag_")
        or col.startswith("pm2_5_rolling_mean_")
        or col.startswith("pm10_rolling_mean_")
    )
]

print("Number of features:", len(feature_columns))
print(feature_columns)

Number of features: 36
['pm2_5_lag_1', 'pm10_lag_1', 'pm2_5_lag_2', 'pm10_lag_2', 'pm2_5_lag_3', 'pm10_lag_3', 'pm2_5_lag_4', 'pm10_lag_4', 'pm2_5_lag_5', 'pm10_lag_5', 'pm2_5_lag_6', 'pm10_lag_6', 'pm2_5_lag_7', 'pm10_lag_7', 'pm2_5_lag_8', 'pm10_lag_8', 'pm2_5_lag_9', 'pm10_lag_9', 'pm2_5_lag_10', 'pm10_lag_10', 'pm2_5_lag_11', 'pm10_lag_11', 'pm2_5_lag_12', 'pm10_lag_12', 'pm2_5_lag_13', 'pm10_lag_13', 'pm2_5_lag_14', 'pm10_lag_14', 'pm2_5_lag_15', 'pm10_lag_15', 'pm2_5_rolling_mean_3', 'pm10_rolling_mean_3', 'pm2_5_rolling_mean_7', 'pm10_rolling_mean_7', 'pm2_5_rolling_mean_15', 'pm10_rolling_mean_15']


In [24]:
final_test = test_features[
    ["date", "pm2_5", "pm10"] + feature_columns
].copy()

output_path = Path(
    "../data/processed/ratnapark/"
    "ratnapark_pm25_pm10_test_15days.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

final_test.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", final_test.shape)

display(final_test.head())

Saved: ..\data\processed\ratnapark\ratnapark_pm25_pm10_test_15days.csv
Shape: (15, 39)


,date,pm2_5,pm10,pm2_5_lag_1,pm10_lag_1,pm2_5_lag_2,pm10_lag_2,pm2_5_lag_3,pm10_lag_3,pm2_5_lag_4,...,pm2_5_lag_14,pm10_lag_14,pm2_5_lag_15,pm10_lag_15,pm2_5_rolling_mean_3,pm10_rolling_mean_3,pm2_5_rolling_mean_7,pm10_rolling_mean_7,pm2_5_rolling_mean_15,pm10_rolling_mean_15
0,2026-08-02 00:00:00+00:00,58.833538,64.022251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-08-03 00:00:00+00:00,62.634817,75.393556,58.833538,64.022251,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-08-04 00:00:00+00:00,58.748684,69.281502,62.634817,75.393556,58.833538,64.022251,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-08-05 00:00:00+00:00,39.631938,49.294396,58.748684,69.281502,62.634817,75.393556,58.833538,64.022251,NaN,...,NaN,NaN,NaN,NaN,60.072347,69.565770,NaN,NaN,NaN,NaN
4,2026-08-06 00:00:00+00:00,41.120997,49.773565,39.631938,49.294396,58.748684,69.281502,62.634817,75.393556,58.833538,...,NaN,NaN,NaN,NaN,53.671813,64.656485,NaN,NaN,NaN,NaN
